# RHIC p+p 200 GeV hard tune: comparison to data

The X-SCAPE ISR chain in vacuum (3dMCGlauber + PythiaIsrGun + iMATTER + Matter + iColored),
tuned to **PHENIX $\pi^0$** (6–20 GeV) and **STAR inclusive jets $R=0.5$** (8.2–52 GeV,
$|\eta|<0.5$ and $0.5<|\eta|<0.9$); **STAR $\pi^0$ and $\eta/\pi^0$** are shown as validation
only. Parameters: $\alpha_s(M_Z)$ (Pythia hard process and MPI), $C_v$ and $Q_0$ (iMATTER and
Matter). Fixed: `PDF:pSet = 20` (iMATTER's NNPDF3.1 NNLO), $\Lambda_\mathrm{QCD}=0.2$,
`probStoUD` 0.217, `MultipartonInteractions:pTmin = 5` (which Pythia lowers to 1.7 GeV at 200 GeV,
see the MPI section), pTHat windows from 5 GeV.

Produced by the scripts of this folder (see `README.md`): `run_runs.py` → `observables.py`
→ `tune.py` → validation runs `best` (the posterior median: the posterior is flat along an
$\alpha_s$–$C_v$ valley, so its MAP on the box edge $C_v = 0.5$ is not a meaningful choice),
`ref` ($\alpha_s=0.15$, $C_v=0.25$, $Q_0=2$: the 5.02 TeV values of arXiv:2407.17443 with
$Q_0$ of its p+p config), the two ends of the valley (`valley_lo`, $C_v=0.16$; `valley_hi`,
$C_v=0.5$) and the MPI checks (`best_mpi4`, `best_nompi`).
**Model results, not vetted physics results.**

In [ ]:
import json, pickle, sys
import numpy as np
import matplotlib.pyplot as plt
import config as C, observables as O, tune as T

data = O.load_data()
summary = json.loads((T.RES / "summary.json").read_text())
design_chi2 = json.loads((T.RES / "design_chi2.json").read_text())
ch = np.load(T.RES / "chains.npz"); chain = ch["chain"]
pp = np.load(T.RES / "posterior_predictive.npz")
runs = {}
REFS = {"pp19": "JETSCAPE PP19", "pythia8_default": "Pythia 8 default (Monash)",
        "pythia8_detroit": "Pythia 8 Detroit (STAR)"}
for name in ("best", "ref", "valley_lo", "valley_hi", "best_mpi4", "best_nompi", *REFS):
    if C.obs_path(name).exists():
        runs[name] = np.load(C.obs_path(name))
print("validation runs:", list(runs), "| posterior samples:", len(chain))
print("MAP:", {k: round(v, 4) for k, v in summary["map"].items()})
print("median:", {k: round(v, 4) for k, v in summary["median"].items()})
print("chi2(MAP, emulated) = %.1f / %d dof" % (summary["chi2_map_emulated"], summary["ndof"]))

## Posterior

In [ ]:
names = T.NAMES
labels = [C.PARAM_LABELS[k] for k in names]
n = len(names)
fig, axes = plt.subplots(n, n, figsize=(8, 8))
for i in range(n):
    for j in range(n):
        ax = axes[i, j]
        if j > i:
            ax.axis("off"); continue
        if i == j:
            ax.hist(chain[:, i], bins=40, range=C.PARAMETERS[names[i]], color="C0", histtype="stepfilled", alpha=0.6)
            ax.axvline(summary["map"][names[i]], color="k", lw=1)
            ax.axvline(summary["median"][names[i]], color="C3", lw=1, ls="--")
            ax.set_yticks([])
        else:
            ax.hist2d(chain[:, j], chain[:, i], bins=40,
                      range=[C.PARAMETERS[names[j]], C.PARAMETERS[names[i]]], cmap="Blues")
            ax.plot(summary["map"][names[j]], summary["map"][names[i]], "k+", ms=10)
            ax.plot(summary["median"][names[j]], summary["median"][names[i]], "x", color="C3", ms=8)
        if i == n - 1: ax.set_xlabel(labels[j])
        if j == 0 and i > 0: ax.set_ylabel(labels[i])
fig.suptitle("posterior (uniform priors on the plotted ranges); + = MAP, red x / dashed = median (the run best)")
fig.tight_layout()
for k in names:
    print(f"{k:8s} MAP {summary['map'][k]:.4f}  median {summary['median'][k]:.4f}  "
          f"68% [{summary['ci68'][k][0]:.4f}, {summary['ci68'][k][1]:.4f}]  "
          f"95% [{summary['ci95'][k][0]:.4f}, {summary['ci95'][k][1]:.4f}]")
print("correlations:\n", np.round(np.array(summary["correlation"]), 2))

## Cross-check: $\chi^2$ of the simulated design points (no emulator)

In [ ]:
X = np.array([[d["params"][k] for k in names] for d in design_chi2])
c2 = np.array([d["chi2"] for d in design_chi2])
fig, axes = plt.subplots(1, n, figsize=(12, 3.3), sharey=True)
for i, ax in enumerate(axes):
    ax.scatter(X[:, i], c2, c=c2, cmap="viridis_r", s=25)
    ax.axvline(summary["map"][names[i]], color="k", lw=1, ls="--")
    ax.set_xlabel(labels[i]); ax.set_yscale("log")
axes[0].set_ylabel(r"$\chi^2$ (%d bins)" % summary["n_bins"])
fig.suptitle("design points; dashed = emulator MAP"); fig.tight_layout()
for d in design_chi2[:8]:
    print(d["point"], {k: round(v, 3) for k, v in d["params"].items()}, "chi2 = %.1f" % d["chi2"])
lo = summary["emulator_loo"]
print("emulator leave-one-out pulls: mean %+.2f, rms %.2f (%d predictions); emulator std scaled by %.2f "
      "in the fit (LOO pull rms 1)" % (lo["pull_mean"], lo["pull_rms"], lo["n"], lo.get("std_scale", 1.0)))

## Data vs model
Points: data (stat ⊕ sys; the normalisation uncertainty is not drawn). Band: posterior predictive of the emulator (68 % / 95 %). Lines: the simulated validation runs. Grey: outside the fit range.

In [ ]:
def centers(o):
    e = np.asarray(data[o]["edges"]); return 0.5 * (e[:, 0] + e[:, 1]), e

def panel(o, ax, rax, pt_min=None, logy=True):
    x, e = centers(o)
    d = np.asarray(data[o]["value"]); err = np.hypot(data[o]["stat"], data[o]["sys"])
    sel = x >= (pt_min if pt_min is not None else 0)
    fitm = T.fit_mask(o, data)
    s = np.exp(pp[o])                                   # (samples, bins)
    lo68, hi68 = np.nanpercentile(s, [16, 84], axis=0)
    lo95, hi95 = np.nanpercentile(s, [2.5, 97.5], axis=0)
    ok = sel & np.isfinite(lo68)
    ax.fill_between(x[ok], lo95[ok], hi95[ok], color="C0", alpha=0.2, step="mid", label="posterior 95%")
    ax.fill_between(x[ok], lo68[ok], hi68[ok], color="C0", alpha=0.4, step="mid", label="posterior 68%")
    rax.fill_between(x[ok], (lo95 / d)[ok], (hi95 / d)[ok], color="C0", alpha=0.2, step="mid")
    rax.fill_between(x[ok], (lo68 / d)[ok], (hi68 / d)[ok], color="C0", alpha=0.4, step="mid")
    for name, col, ls in (("best", "C3", "-"), ("ref", "C2", "-"), ("valley_lo", "C1", ":"),
                          ("valley_hi", "C4", ":")):
        if name in runs:
            v, ve = O.combine(runs[name], o, data)
            m = sel & (v > 0)
            ax.errorbar(x[m], v[m], ve[m], fmt=ls, color=col, lw=1.2, label=f"simulated: {name}")
            rax.errorbar(x[m], (v / d)[m], (ve / d)[m], fmt=ls, color=col, lw=1.2)
    ax.errorbar(x[sel & fitm], d[sel & fitm], err[sel & fitm], fmt="ko", ms=3.5, label="data (fit)")
    ax.errorbar(x[sel & ~fitm], d[sel & ~fitm], err[sel & ~fitm], fmt="o", color="0.6", ms=3.5,
                label="data (not fitted)")
    rax.errorbar(x[sel], np.ones(sel.sum()), (err / d)[sel], fmt="none", ecolor="k", lw=0.8)
    rax.axhline(1, color="k", lw=0.6)
    if logy: ax.set_yscale("log")
    ax.set_title(data[o]["label"], fontsize=10); rax.set_xlabel(r"$p_T$ [GeV]")
    rax.set_ylabel("model / data"); rax.set_ylim(0.3, 1.9)

fig, axs = plt.subplots(2, 3, figsize=(15, 6.5), gridspec_kw={"height_ratios": [2.2, 1]}, sharex="col")
for k, (o, pmin) in enumerate((("phenix_pi0", 4.0), ("star_jets_eta0", 6.0), ("star_jets_eta1", 6.0))):
    panel(o, axs[0, k], axs[1, k], pmin)
axs[0, 0].set_ylabel(r"$E\,d^3\sigma/dp^3$ [mb GeV$^{-2}$]"); axs[0, 1].set_ylabel(r"$d^2\sigma/dp_T d\eta$ [pb/GeV]")
axs[0, 0].legend(fontsize=7); fig.tight_layout()

## Validation (not in the fit): STAR $\pi^0$ and $\eta/\pi^0$, $0<\eta<1$

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(11, 6.5), gridspec_kw={"height_ratios": [2.2, 1]}, sharex="col")
panel("star_pi0", axs[0, 0], axs[1, 0], 4.0)
panel("star_eta_pi0", axs[0, 1], axs[1, 1], 4.0, logy=False)
axs[0, 0].set_ylabel(r"$E\,d^3\sigma/dp^3$ [mb GeV$^{-2}$]"); axs[0, 1].set_ylabel(r"$\eta/\pi^0$")
axs[0, 1].set_ylim(0, 1.0); axs[0, 0].legend(fontsize=7); fig.tight_layout()

## Other p+p models
For orientation, the same observables from models that are not tuned here (`refs/run_refs.py`,
600k events each, the same pTHat windows, π⁰ and η undecayed):
- **JETSCAPE PP19** (arXiv:1910.05481): PythiaGun with ISR and MPI (Pythia defaults, FSR off),
  partons above 2 GeV showered by Matter in vacuum ($Q_0 = 1$ GeV, $C_v = 0.25$), Colorless
  hadronization (`refs/pp19_200.xml`). Tuned mainly to LHC jets; the only JETSCAPE p+p tune that
  covers 200 GeV.
- **Pythia 8 default**: standalone Pythia 8.312 as it comes (Monash 2013).
- **Pythia 8 Detroit**: STAR's 200 GeV tune (PRD 105, 016011), the MPI and colour-reconnection
  settings of `refs/pythia_ref.py`.

Shown as model / data with the tune's `best` run; the χ² table above includes them.

In [ ]:
models = [("best", "X-SCAPE tune (best)", "C3", "-")] + [
    (n, lab, col, "--") for (n, lab), col in zip(REFS.items(), ("C0", "C1", "C2"))]
obs = [("phenix_pi0", 4.0), ("star_jets_eta0", 6.0), ("star_jets_eta1", 6.0), ("star_pi0", 4.0), ("star_eta_pi0", 4.0)]
fig, axs = plt.subplots(1, len(obs), figsize=(20, 3.8))
for ax, (o, pmin) in zip(axs, obs):
    x, e = centers(o)
    d = np.asarray(data[o]["value"]); rel = np.hypot(data[o]["stat"], data[o]["sys"]) / d
    sel = x >= pmin
    ax.fill_between(x[sel], 1 - rel[sel], 1 + rel[sel], color="0.85", step="mid", label="data stat⊕sys")
    if o in C.FIT:
        lo_, hi_ = C.FIT[o]; ax.axvspan(e[0, 0], lo_, color="0.95", zorder=0)
    for name, lab, col, ls in models:
        if name not in runs: continue
        v, ve = O.combine(runs[name], o, data)
        m = sel & (v > 0)
        ax.errorbar(x[m], (v / d)[m], (ve / d)[m], fmt=ls, marker=".", color=col, lw=1.2, label=lab)
    ax.axhline(1, color="k", lw=0.6); ax.set_ylim(0.0, 2.5)
    ax.set_title(data[o]["label"], fontsize=9); ax.set_xlabel(r"$p_T$ [GeV]")
axs[0].set_ylabel("model / data"); axs[0].legend(fontsize=7); fig.tight_layout()
for name, lab, col, ls in models:
    if name not in runs: continue
    parts = []
    for o in ("phenix_pi0", "star_jets_eta0", "star_jets_eta1", "star_pi0"):
        x, e = centers(o); v, _ = O.combine(runs[name], o, data); d = np.asarray(data[o]["value"])
        m = T.fit_mask(o, data) if o in C.FIT else (x > 6)
        parts.append(f"{o} {np.exp(np.mean(np.log(v[m] / d[m]))):.2f}")
    print(f"{lab:28s} mean model/data: " + ", ".join(parts))


## $\chi^2$ per data set of the simulated validation runs

In [ ]:
like_blocks = []
for o in C.FIT:
    m = T.fit_mask(o, data)
    d = np.asarray(data[o]["value"])[m]
    rel2 = (np.asarray(data[o]["stat"])[m] ** 2 + np.asarray(data[o]["sys"])[m] ** 2) / d ** 2
    like_blocks.append((o, m, d, rel2, data[o]["norm"] or 0.0))
for name, ob in runs.items():
    tot = 0.0; nb = 0
    for o, m, d, rel2, norm in like_blocks:
        v, ve = O.combine(ob, o, data)
        r = np.log(v[m] / d)
        Cm = np.diag(rel2 + (ve[m] / v[m]) ** 2) + norm ** 2
        c = float(r @ np.linalg.solve(Cm, r)); tot += c; nb += m.sum()
        print(f"{name:10s} {o:16s} chi2 = {c:6.1f} / {m.sum()} bins")
    print(f"{name:10s} total            chi2 = {tot:6.1f} / {nb} bins\n")

## The $\alpha_s$–$C_v$ valley: $C_v$ is not constrained by p+p
Profile of the emulated likelihood in $C_v$ ($\alpha_s$ and $Q_0$ re-fitted at every $C_v$), and the
$\chi^2$ of the simulated runs at the two ends of the valley and at the median. A larger $C_v$ starts
the shower at higher virtuality (more radiation, softer spectra); a larger $\alpha_s$ compensates.
The O+O medium modification depends on $C_v$, so p+p alone does not fix it.

In [ ]:
from scipy.optimize import minimize
sys.modules["__main__"].Emulator = T.Emulator          # unpickle tune.py's class
emu = pickle.load(open(T.RES / "emulator.pkl", "rb"))["emulator"]
like = T.Likelihood(data, emu)
cvs = np.linspace(*C.PARAMETERS["Cv"], 12); prof = []
for cv in cvs:
    o = minimize(lambda y: -like.loglike(np.array([y[0], cv, y[1]])), [0.153, 1.5], method="Nelder-Mead")
    x = np.array([o.x[0], cv, o.x[1]]); r, v = like.residual(x)
    prof.append((o.x[0], o.x[1], -o.fun, like.chi2(r, v)))
prof = np.array(prof)
fig, axs = plt.subplots(1, 3, figsize=(13, 3.3))
axs[0].plot(cvs, prof[:, 2] - prof[:, 2].max(), "o-"); axs[0].set_ylabel(r"$\Delta\ln L$ (profile)")
axs[1].plot(cvs, prof[:, 0], "o-"); axs[1].set_ylabel(r"best $\alpha_s(M_Z)$")
axs[2].plot(cvs, prof[:, 1], "o-"); axs[2].set_ylabel(r"best $Q_0$ [GeV]")
for ax in axs: ax.set_xlabel(r"$C_v$")
fig.tight_layout()
for cv, (a_s, q0, ll, c2) in zip(cvs, prof):
    print(f"Cv {cv:.3f}: alpha_s {a_s:.4f}, Q0 {q0:.2f}, chi2 (emulated) {c2:.1f}")


## Is fixing the MPI cut justified?
**The setting `MultipartonInteractions:pTmin = 5` is not a 5 GeV cut at 200 GeV.** Pythia's MPI
needs $\sigma_\mathrm{int}(p_T > p_{T,\min}) > 1.1\,\sigma_\mathrm{ND}$ (28.5 mb); above 5 GeV it is
0.33 mb, so Pythia lowers $p_{T0}$ and $p_{T,\min}$ in steps of 0.9 (Pythia 8.312
`MultipartonInteractions.cc`, init) until it holds: $p_{T0}$ 1.06 → 0.37 GeV, $p_{T,\min}$ 5 → 1.7 GeV
(the job logs show the steps). Any setting from 2 to 7 GeV ends near 1.6–1.8 GeV with ~0.2–0.55
extra MPI per event; `best_mpi4` is therefore the same model as `best`, and MPI off is the only
real variation. At the best point, the same chain with `pTmin = 4` (`best_mpi4`) and with MPI
switched off (`best_nompi`), as ratios to `best`
(statistical errors of both runs). If the shifts stay well inside the data errors (grey: the
data's stat ⊕ sys relative to their value), the fit data cannot constrain the MPI cut.

In [ ]:
variants = [v for v in ("best_mpi4", "best_nompi") if v in runs]
if "best" not in runs or not variants:
    print("needs the runs best, best_mpi4, best_nompi")
else:
    obs = ("phenix_pi0", "star_jets_eta0", "star_jets_eta1", "star_pi0")
    fig, axs = plt.subplots(1, len(obs), figsize=(16, 3.4))
    for ax, o in zip(axs, obs):
        x, e = centers(o)
        b, be = O.combine(runs["best"], o, data)
        d = np.asarray(data[o]["value"]); rel = np.hypot(data[o]["stat"], data[o]["sys"]) / d
        fitm = T.fit_mask(o, data) if o in C.FIT else (x > 6)
        m = fitm & (b > 0)
        ax.fill_between(x[m], 1 - rel[m], 1 + rel[m], color="0.85", step="mid", label="data stat⊕sys")
        for v, col in zip(variants, ("C1", "C4")):
            vv, ve = O.combine(runs[v], o, data)
            r = vv / b; re = r * np.hypot(ve / vv, be / b)
            ax.errorbar(x[m], r[m], re[m], fmt="o-", ms=3, color=col, label=v)
            w = 1 / re[m] ** 2
            print(f"{o:15s} {v:11s} mean ratio {np.sum(w * r[m]) / np.sum(w):.3f} "
                  f"+- {1 / np.sqrt(np.sum(w)):.3f}")
        ax.axhline(1, color="k", lw=0.6); ax.set_ylim(0.6, 1.4)
        ax.set_title(data[o]["label"], fontsize=9); ax.set_xlabel(r"$p_T$ [GeV]")
    axs[0].set_ylabel("variant / best"); axs[0].legend(fontsize=7); fig.tight_layout()

## Underlying event: STAR transverse density vs leading jet (not in the fit)
STAR, PRD 101, 052004 (HEPData ins1771348): leading jet anti-$k_T$ $R=0.6$, $|\eta_\mathrm{jet}|<0.4$;
charged particles $p_T>0.2$ GeV, $|\eta|<1$, transverse region $60°<|\Delta\phi|<120°$. From
`ue_check.py` on the X-SCAPE vacuum runs: **no strings and no hydro**, so the model UE is beam remnants,
ISR/FSR and the MPI (effective cut 1.7 GeV); whatever the soft sector adds comes on top. Leading-jet
bins below ~9 GeV are incomplete (no $\hat p_T<5$ GeV: 21 % of the 5–7 GeV bin missing in `trunc`),
for the references as well. The Pythia references have their full MPI and hadronization (Pythia's own
UE); PP19 showers only partons above 2 GeV and closes strings with Colorless's remnants.

In [ ]:
ue = json.loads((T.RES / "ue_check.json").read_text())
st = ue["star"]; e = np.asarray(st["edges"]); xc = 0.5 * (e[:-1] + e[1:])
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
for ax, key, skey, lab in ((axs[0], "n_transverse", "transverse", "$p_T>0.2$ GeV"),
                           (axs[1], "n_trans05", "transverse05", "$p_T>0.5$ GeV")):
    d = st[skey]; err = np.hypot(d["stat"], d["sys"])
    ax.errorbar(xc, d["value"], err, fmt="ko", label="STAR")
    for run, col, ls in (("best", "C3", "-"), ("best_nompi", "C4", "--"), ("ref", "C2", "-"),
                         ("valley_lo", "C1", ":"), ("valley_hi", "C0", ":"),
                         ("pp19", "C5", "-."), ("pythia8_default", "C7", "-."),
                         ("pythia8_detroit", "C8", "-.")):
        if run in ue["runs"]:
            y = [r[key] for r in ue["runs"][run]]
            ax.plot(xc, y, ls, color=col, marker=".", label=run)
    ax.axvspan(5, 9, color="0.9", zorder=0)
    ax.set_xlabel("leading jet $p_T$ [GeV]"); ax.set_ylabel(r"transverse $dN_{ch}/d\eta d\phi$ (" + lab + ")")
    ax.set_ylim(0, 1.0)
axs[0].legend(fontsize=8); fig.tight_layout()
print("C_v: valley_lo 0.16, ref 0.25, best 0.32, valley_hi 0.50")
for run in ue["runs"]:
    if run.endswith("_low_share"): continue
    print(f"{run:11s}", " ".join(f"{r['n_transverse']:.3f}" for r in ue["runs"][run]))
print("STAR       ", " ".join(f"{v:.3f}" for v in st["transverse"]["value"]))


## Notes
- Model jets: anti-$k_T$ $R=0.5$ on all visible final particles with $p_T>0.2$ GeV, at least one
  charged constituent, underlying event subtracted with STAR's off-axis cones ($A=\pi R^2$).
  In vacuum mode there is no bulk: the remaining underlying event is MPI (effective cut 1.7 GeV),
  ISR/FSR and beam remnants.
- $\pi^0$ and $\eta$ are kept undecayed in the model; invariant cross sections use $|y|$ windows.
- Systematic uncertainties are treated as uncorrelated between bins; normalisation
  uncertainties (PHENIX 9.7 %, STAR 5.6 %) as fully correlated within a data set.
- The lowest pTHat window starts at 5 GeV; `truncation_check.py` shows that this cannot reach
  $\pi^0$ above 5.7 GeV or jets above 8.2 GeV, hence the fit ranges.